# Blood Work Analysis Pipeline
A two-stage LLM pipeline:
- **Stage 1:** Extract and flag abnormal values from a blood report
- **Stage 2:** Generate a health summary and Indian diet plan based on flagged values

In [1]:
# from dotenv import load_dotenv
# from langchain_google_genai import ChatGoogleGenerativeAI

# load_dotenv()

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

local_api_key = os.environ["LOCAL_LLM_API_KEY"]
local_base_url = os.environ["LOCAL_LLM_BASE_URL"]

def get_local_llm(model, **kwargs):
    return ChatGoogleGenerativeAI(
        model=model,
        api_key=local_api_key,
        base_url=local_base_url,
        **kwargs,
    )
llm = get_local_llm(model="gemini-auto")

## Sample Blood Report

In [2]:
with open("blood_work.txt", "r") as f:
    blood_report = f.read()

print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5â€“17.5)
Hematocrit:        44%          


## Stage 1 — Extract and Flag Abnormal Values

In [3]:
extraction_prompt = f"""
You are a medical data extraction assistant.

From the blood report below, extract ALL test values and classify each one as HIGH, LOW, or NORMAL 
based on the reference ranges provided in the report.

Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text

print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


=== STAGE 1: EXTRACTED VALUES ===
- Hemoglobin: 15.1 g/dL | Status: NORMAL | Reference: 13.5–17.5 g/dL
- Hematocrit: 44% | Status: NORMAL | Reference: 41–53%
- WBC: 6.8 x10^3/uL | Status: NORMAL | Reference: 4.5–11.0 x10^3/uL
- Platelets: 220 x10^3/uL | Status: NORMAL | Reference: 150–400 x10^3/uL
- Total Cholesterol: 238 mg/dL | Status: HIGH | Reference: <200 mg/dL
- LDL Cholesterol: 162 mg/dL | Status: HIGH | Reference: <100 mg/dL
- HDL Cholesterol: 36 mg/dL | Status: LOW | Reference: >40 mg/dL
- Triglycerides: 188 mg/dL | Status: HIGH | Reference: <150 mg/dL
- Glucose (Fasting): 92 mg/dL | Status: NORMAL | Reference: 70–99 mg/dL
- HbA1c: 5.3% | Status: NORMAL | Reference: <5.7%
- Creatinine: 1.0 mg/dL | Status: NORMAL | Reference: 0.7–1.3 mg/dL
- eGFR: 82 mL/min | Status: NORMAL | Reference: >60 mL/min
- ALT: 28 U/L | Status: NORMAL | Reference: 7–40 U/L
- AST: 25 U/L | Status: NORMAL | Reference: 10–40 U/L
- Bilirubin Total: 0.8 mg/dL | Status: NORMAL | Reference: 0.2–1.2 mg/dL


## Stage 2 — Health Summary and Indian Diet Plan

In [4]:
diet_prompt = f"""
You are a clinical nutritionist specializing in Indian dietary habits.

Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines explaining the patient's condition in simple language
2. A short, practical Indian diet plan having only two sections (1) Foods to avoid (2) Foods to eat more of. 
   Do not include any other sections in diet plan.

Blood Work Analysis:
{extracted_values}
"""

diet_response = llm.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.text)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
## 1. Health Summary

Your blood work shows normal blood sugar, kidney function, liver enzymes, and blood counts, which is great news. However, your lipid panel indicates **dyslipidemia**, characterized by elevated total cholesterol, high LDL ("bad") cholesterol, high triglycerides, and low HDL ("good") cholesterol. This pattern increases the risk of plaque buildup in your arteries and cardiovascular complications over time. Fortunately, these parameters respond exceptionally well to targeted dietary and lifestyle modifications. Focusing on heart-healthy fats, soluble fiber, and reduced refined carbohydrates will help bring these numbers back into the optimal range.

---

## 2. Indian Diet Plan

### (1) Foods to Avoid
* **Refined Carbohydrates & Sugars:** White bread, maida-based items (naan, maida biscuits), sweets, commercially packaged baked goods, and sugary beverages.
* **Trans & Saturated Fats:** Vanaspati, margarine, excessive butter, 